In [0]:
file_path = "/databricks-datasets/learning-spark-v2/flights/departuredelays.csv"

raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(raw_df.limit(5))

# Save as a managed Delta table
raw_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_flights")

print("Bronze table saved successfully as a managed table!")

date,delay,distance,origin,destination
1011245,6,602,ABE,ATL
1020600,-8,369,ABE,DTW
1021245,-2,602,ABE,ATL
1020605,-4,602,ABE,ATL
1031245,-4,602,ABE,ATL


Bronze table saved successfully as a managed table!


In [0]:
# Read from the bronze table
bronze_df = spark.read.table("bronze_flights")

# Clean data (drop nulls and filter valid records)
silver_df = bronze_df.dropna().filter(bronze_df["delay"] > -100)

display(silver_df.limit(5))

# Save as a managed silver table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_flights")

print("Silver table cleaned and saved successfully!")

date,delay,distance,origin,destination
1011245,6,602,ABE,ATL
1020600,-8,369,ABE,DTW
1021245,-2,602,ABE,ATL
1020605,-4,602,ABE,ATL
1031245,-4,602,ABE,ATL


Silver table cleaned and saved successfully!


In [0]:
from pyspark.sql.functions import col, avg, desc

# Read from the silver table
silver_df = spark.read.table("silver_flights")

# Calculate average delay by origin airport
gold_df = silver_df.groupBy("origin") \
    .agg(avg("delay").alias("avg_delay")) \
    .orderBy(desc("avg_delay"))

display(gold_df)

# Save as a managed gold table
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_flights_summary")

print("Gold revenue aggregation table created successfully!")

origin,avg_delay
GUM,33.87777777777778
LSE,26.532467532467532
MQT,23.87012987012987
EGE,20.57012542759407
ROA,19.885106382978723
MDW,19.657658556043078
BTV,18.7246192893401
ORD,18.588917606028524
IAD,18.40343803056027
SCE,17.91616766467066


Gold revenue aggregation table created successfully!


In [0]:

%sql
SELECT 
    origin, 
    ROUND(avg_delay, 2) AS formatted_avg_delay 
FROM 
    gold_flights_summary 
ORDER BY 
    avg_delay DESC 
LIMIT 10;

origin,formatted_avg_delay
GUM,33.88
LSE,26.53
MQT,23.87
EGE,20.57
ROA,19.89
MDW,19.66
BTV,18.72
ORD,18.59
IAD,18.4
SCE,17.92


In [0]:
%sql
SELECT 
    s.date,
    s.origin,
    s.destination,
    s.delay,
    g.avg_delay AS origin_avg_delay
FROM 
    silver_flights s
JOIN 
    gold_flights_summary g 
ON 
    s.origin = g.origin
WHERE 
    s.delay > 60
ORDER BY 
    s.delay DESC
LIMIT 10;

date,origin,destination,delay,origin_avg_delay
3090615,TPA,DFW,1642,14.105069070475299
2190925,SFO,ORD,1638,12.705974723298635
2021245,FLL,DFW,1636,16.514384094190824
3020700,RSW,ORD,1592,12.856248009342817
1180805,BNA,DFW,1560,15.454962499089783
3031210,PDX,DFW,1553,6.68213205058577
3070645,CLE,DFW,1543,14.035917207792208
2210630,MCO,ORD,1511,15.71963409034719
1300915,EGE,JFK,1500,20.57012542759407
1150715,ONT,DFW,1496,8.740960163432074


In [0]:
# Query using PySpark SQL engine
high_delays_df = spark.sql("""
    SELECT origin, avg_delay 
    FROM gold_flights_summary 
    WHERE avg_delay > 15
    ORDER BY avg_delay DESC
""")

# Display the query results
display(high_delays_df)

origin,avg_delay
GUM,33.87777777777778
LSE,26.532467532467532
MQT,23.87012987012987
EGE,20.57012542759407
ROA,19.885106382978723
MDW,19.657658556043078
BTV,18.7246192893401
ORD,18.588917606028524
IAD,18.40343803056027
SCE,17.91616766467066


In [0]:
%sql
DESCRIBE HISTORY gold_flights_summary;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-01T13:59:10.000Z,74811284307273,lokanath2625@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2245462674996558),7a16105c-5a37-4d64-8968-83a0e870ed0b,1101-135455-wvupbjga-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 255, numOutputBytes -> 3511)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

# 1. Read data from your Silver Delta table into a Spark DataFrame
silver_df = spark.read.table("silver_flights")

# 2. Prepare a sample for training (convert to Pandas DataFrame for Scikit-Learn)
# We create a binary target: 1 if delay > 15 minutes, else 0
pdf = silver_df.select("distance", "delay").withColumn("is_delayed", (col("delay") > 15).cast("int")).limit(50000).toPandas()

X = pdf[["distance"]]
y = pdf["is_delayed"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Start an MLflow run to track training
with mlflow.start_run(run_name="flight_delay_prediction_model") as run:
    
    # Define model parameters
    c_param = 1.0
    
    # Train a Logistic Regression model
    model = LogisticRegression(C=c_param, random_state=42)
    model.fit(X_train, y_train)
    
    # Make predictions
    predictions = model.predict(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, predictions)
    
    # Log parameters and metrics to MLflow
    mlflow.log_param("regularization_C", c_param)
    mlflow.log_metric("accuracy", acc)
    
    # Log the trained model artifact
    mlflow.sklearn.log_model(model, "flight_delay_model")
    
    print(f"Model trained successfully! MLflow Run ID: {run.info.run_id}")
    print(f"Model Accuracy: {acc:.4f}")

2026/11/01 14:01:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-43fd5ff3-42a9.cloud.databricks.com/ml/experiments/2245462674996558/models/m-e8beb25058a84701b4fb642cb6841cd6?o=7474654066756921
2026/11/01 14:02:00 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


Model trained successfully! MLflow Run ID: dab2ec4c9bcd4e05baf8f2be5f2ee411
Model Accuracy: 0.7435


In [0]:
import mlflow
import mlflow.sklearn
from pyspark.sql.functions import col
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare data from your Silver Delta table for classification
# Predict whether a flight is heavily delayed (> 15 mins) based on distance
pdf = spark.read.table("silver_flights") \
    .select("distance", "delay") \
    .withColumn("is_delayed", (col("delay") > 15).cast("int")) \
    .limit(50000) \
    .toPandas()

# Drop any potential null rows
pdf = pdf.dropna()

X = pdf[["distance"]]
y = pdf["is_delayed"]

# 2. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Enable MLflow tracking and train the model
with mlflow.start_run(run_name="flight_delay_logistic_regression") as run:
    # Define hyperparameters
    reg_c = 1.0
    
    # Train Scikit-Learn Logistic Regression model
    model = LogisticRegression(C=reg_c, random_state=42)
    model.fit(X_train, y_train)
    
    # Make predictions and evaluate accuracy
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # Log parameters and metrics to MLflow
    mlflow.log_param("regularization_C", reg_c)
    mlflow.log_metric("accuracy", acc)
    
    # Log the trained model artifact
    mlflow.sklearn.log_model(model, "flight_delay_model")
    
    print(f"Model trained successfully! Test Accuracy: {acc:.4f}")
    print(f"MLflow Run ID: {run.info.run_id}")

{"ts": "2026-11-01 14:41:09.610", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjI0NTQ2MjY3NDk5NjU1OBABIAEyJDAxYTE5NzhlLTgyODctNzQxNi1iMWQ0LWU5ZDE4ZGU2MDllOTokYWYwYmQ2ZWMtOTNiMi0zYmQ0LTk2ZDYtNzdlZDI2YzNjYmI4SgwI/Z6d1wYQwJuXkgJQAVgBYAFouZrfo4jFow2IAQA=.", "context": {}}
{"ts": "2026-11-01 14:41:09.610", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjI0NTQ2MjY3NDk5NjU1OBABIAEyJDAxYTE5NzhlLTgyODctNzQxNi1iMWQ0LWU5ZDE4ZGU2MDllOTokYWYwYmQ2ZWMtOTNiMi0zYmQ0LTk2ZDYtNzdlZDI2YzNjYmI4SgwI/Z6d1wYQwJuXkgJQAVgBYAFouZrfo4jFow2IAQA=.", "context": {}}
{"ts": "2026-11-01 14:41:09.610", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjI0NTQ2MjY3NDk5NjU1OBABIAEyJDAxYTE5NzhlLTgyODctNzQxNi1iMWQ0LWU5ZDE4ZGU2MDllOTokYWYwYmQ2ZWMtOTNiMi0zYmQ0LTk2ZDYtNzdlZDI2

Model trained successfully! Test Accuracy: 0.7435
MLflow Run ID: 3e59f577d40d450a85b1d1d8f772dba2
